In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from safetensors.torch import load_file
from PIL import Image
import cv2

# Initialize the exact same architecture used in training
class VideoDeepfakeEfficientNet(nn.Module):
    def __init__(self):
        super(VideoDeepfakeEfficientNet, self).__init__()
        self.efficientnet = models.efficientnet_b0()
        num_features = self.efficientnet.classifier[1].in_features
        self.efficientnet.classifier = nn.Sequential(
            nn.Dropout(p=0.5, inplace=True),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256, 1) 
        )
    def forward(self, x):
        return self.efficientnet(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
video_model = VideoDeepfakeEfficientNet().to(device)

# Load the safetensors weights
MODEL_PATH = '/projects/Projects/ai-phishing-defense/models/final/video_detector/model.safetensors'
video_model.load_state_dict(load_file(MODEL_PATH))
video_model.eval()

# Standard ImageNet transforms required by EfficientNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict_video(video_path):
    """
    Inference function for Module 3: Synthetic Video Detector.
    Extracts the middle frame of a video and evaluates it.
    """
    try:
        # Extract a single frame (e.g., the middle frame) using OpenCV
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
        ret, frame = cap.read()
        cap.release()
        
        if not ret:
            return {'error': 'Failed to read video frame.'}

        # Convert OpenCV BGR to RGB, then to PIL Image for standard transforms
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(frame_rgb)
        
        # Transform and add batch dimension: [1, 3, 224, 224]
        input_tensor = transform(pil_img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = video_model(input_tensor)
            prob = torch.sigmoid(output).item()
            
        is_spoof = prob >= 0.5
        label = 'Synthetic' if is_spoof else 'Genuine'
        confidence = prob if is_spoof else (1.0 - prob)
        
        return {
            'label': label,
            'confidence': round(confidence, 4)
        }
        
    except Exception as e:
        return {'error': f"Failed to process video: {str(e)}"}